## Multi-Task PCL Detection

Category-informed architecture: shared ALBERT-v2-large encoder (24 layers, 1024-dim, parameter sharing) with two heads trained jointly.
The 7-category predictions feed directly into the binary classifier as features,
so the model learns "what type of PCL → is this PCL?" end-to-end.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModel,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from pcl_tf.focal_loss import FocalLoss

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")  # better tensor core utilization

In [2]:
# --- CONFIGURATION ---
MODEL_CHECKPOINT = "albert/albert-large-v2"     # 24 layers, 1024-dim hidden, ~18M params (parameter sharing)
CACHE_DIR = "./models_cache"
MAX_LENGTH = 128                   # p95 is ~70-100 tokens — 128 is sufficient
BATCH_SIZE = 32                    # ALBERT shares params → low VRAM; big batch saturates GPU
GRAD_ACCUMULATION = 2              # no accumulation — batch 64 is already large enough
NUM_EPOCHS = 15
LR = 1e-5                         # lower LR for deeper 24-layer model
FOCAL_ALPHA = 0.5                  # reduced from 0.75 — lowers initial loss magnitude
FOCAL_GAMMA = 1.5                  # increased — stronger focus on hard examples once model calibrates
AUX_LOSS_WEIGHT = 0.2              # reduced from 0.3 — less category loss contribution to total
NUM_CATEGORIES = 7
DROPOUT = 0.2
WEIGHT_DECAY = 0.01               # ALBERT uses lighter weight decay
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


## Data extract from categorical span information

In [3]:
# --- DATA LOADING ---

# 1. Binary + text data (all samples)
df = pd.read_csv("data/dontpatronizeme_pcl_cleaned.csv")
df = df.dropna(subset=["text", "par_id"])
df["par_id"] = df["par_id"].astype(int)

# 2. Span-level category annotations → per-paragraph multi-label vectors
#    Format: par_id \t art_id \t text \t keyword \t country_code \t
#            span_start \t span_finish \t span_text \t pcl_category \t num_annotators
cats_df = pd.read_csv(
    "data/dontpatronizeme_categories.tsv",
    sep="\t", header=None, skiprows=4, engine="python",
    names=["par_id", "art_id", "text", "keyword", "country_code",
           "span_start", "span_finish", "span_text", "pcl_category",
           "num_annotators"],
)
cats_df["par_id"] = cats_df["par_id"].astype(int)

# Build a 7-dim one-hot vector per par_id from the pcl_category column
CATEGORIES = sorted(cats_df["pcl_category"].unique())
print(f"PCL categories ({len(CATEGORIES)}): {CATEGORIES}")

cat_dummies = pd.get_dummies(cats_df[["par_id", "pcl_category"]], columns=["pcl_category"], prefix="", prefix_sep="")
cat_vectors = cat_dummies.groupby("par_id")[CATEGORIES].max().astype(int)  # bool → int
cat_vectors = cat_vectors.reset_index()
cat_vectors["multi_label"] = cat_vectors[CATEGORIES].values.tolist()

# 3. Merge: text + binary label + multi-label vector
# Note: df already has a "label" column, so we use "multi_label" to avoid collision
merged = pd.merge(df, cat_vectors[["par_id", "multi_label"]], on="par_id", how="left")

# Rows without a category entry are non-PCL (all zeros)
merged["multi_label"] = merged["multi_label"].apply(lambda x: x if isinstance(x, list) else [0] * len(CATEGORIES))
merged["pcl_binary"] = merged["pcl_binary"].astype(int)

print(f"Merged dataset size: {len(merged)}")
print(f"  PCL positive: {merged['pcl_binary'].sum()}  |  negative: {(merged['pcl_binary'] == 0).sum()}")

PCL categories (7): ['Authority_voice', 'Compassion', 'Metaphors', 'Presupposition', 'Shallow_solution', 'The_poorer_the_merrier', 'Unbalanced_power_relations']
Merged dataset size: 10468
  PCL positive: 993  |  negative: 9475


In [4]:
# --- DATASET & TOKENIZER ---

class MultiTaskPCLDataset(Dataset):
    """Fully pre-computed dataset — zero per-item conversion overhead.
    Pre-converts everything to tensors for faster collation."""
    def __init__(self, texts, binary_labels, cat_labels, tokenizer, max_len):
        enc = tokenizer(texts, truncation=True, max_length=max_len)
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.token_type_ids = enc.get("token_type_ids")
        # Pre-tensorize labels — avoids Python→Tensor conversion per batch
        self.binary_labels = torch.tensor(binary_labels, dtype=torch.long)
        self.cat_labels = torch.tensor(cat_labels, dtype=torch.float32)

    def __len__(self):
        return len(self.binary_labels)

    def __getitem__(self, idx):
        item = {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.binary_labels[idx],       # tensor scalar
            "cat_labels": self.cat_labels[idx],       # 1D tensor (7,)
        }
        if self.token_type_ids is not None:
            item["token_type_ids"] = self.token_type_ids[idx]
        return item


tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, cache_dir=CACHE_DIR)

# Aligned train/val split
train_idx, val_idx = train_test_split(
    range(len(merged)), test_size=0.1, random_state=42, stratify=merged["pcl_binary"]
)

def _make_multitask_datasets():
    texts = merged["text"].tolist()
    binary = merged["pcl_binary"].tolist()
    cats = merged["multi_label"].tolist()

    def _select(idxs):
        return [texts[i] for i in idxs], [binary[i] for i in idxs], [cats[i] for i in idxs]

    tr_texts, tr_bin, tr_cat = _select(train_idx)
    va_texts, va_bin, va_cat = _select(val_idx)

    print(f"  Pre-tokenizing {len(tr_texts)} train + {len(va_texts)} val texts...")
    train_ds = MultiTaskPCLDataset(tr_texts, tr_bin, tr_cat, tokenizer, MAX_LENGTH)
    val_ds   = MultiTaskPCLDataset(va_texts, va_bin, va_cat, tokenizer, MAX_LENGTH)
    return train_ds, val_ds, tr_bin

train_ds, val_ds, train_binary_labels = _make_multitask_datasets()

# --- Weighted sampler to handle class imbalance ---
class_counts = np.bincount(train_binary_labels)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[l] for l in train_binary_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

lengths = [len(ids) for ids in train_ds.input_ids]
print(f"\nToken length stats: mean={np.mean(lengths):.0f}, median={np.median(lengths):.0f}, "
      f"max={max(lengths)}, p95={np.percentile(lengths, 95):.0f}")
print(f"Train/val: {len(train_ds)}/{len(val_ds)}")
print(f"Class balance — neg: {class_counts[0]}, pos: {class_counts[1]}, "
      f"ratio: {class_counts[0]/class_counts[1]:.1f}:1")
print(f"WeightedRandomSampler active — effective 50/50 class balance per epoch.")

  Pre-tokenizing 9421 train + 1047 val texts...

Token length stats: mean=61, median=55, max=128, p95=128
Train/val: 9421/1047
Class balance — neg: 8527, pos: 894, ratio: 9.5:1
WeightedRandomSampler active — effective 50/50 class balance per epoch.


In [6]:
# --- MULTI-TASK MODEL ---
from transformers import AutoConfig

class MultiTaskModel(nn.Module):
    """Encoder with two heads trained jointly:
    1) category_head: [CLS] → 7 PCL category logits (auxiliary signal)
    2) binary_head:   concat([CLS], category_logits) → binary PCL prediction
    The category predictions are a *feature* for the binary decision."""

    def __init__(self, checkpoint=None, num_categories=7, dropout=0.1, cache_dir=None,
                 encoder=None):
        super().__init__()
        # Accept pre-built encoder (cached) or load from checkpoint
        if encoder is not None:
            self.encoder = encoder
        else:
            self.encoder = AutoModel.from_pretrained(checkpoint, cache_dir=cache_dir)
        h = self.encoder.config.hidden_size  # 1024 for ALBERT-large

        self.dropout = nn.Dropout(dropout)

        # Auxiliary: predict which PCL categories are present
        self.category_head = nn.Linear(h, num_categories)

        # Main: binary PCL detection, informed by category logits
        self.binary_head = nn.Sequential(
            nn.Linear(h + num_categories, h // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(h // 2, 2),
        )

    def gradient_checkpointing_enable(self, **kwargs):
        self.encoder.gradient_checkpointing_enable(**kwargs)

    def gradient_checkpointing_disable(self):
        self.encoder.gradient_checkpointing_disable()

    def forward(self, input_ids, attention_mask, token_type_ids=None, **kwargs):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        cls = self.dropout(outputs.last_hidden_state[:, 0])  # [CLS] pooling

        cat_logits = self.category_head(cls)                       # (B, 7)
        binary_input = torch.cat([cls, cat_logits], dim=-1)        # (B, 1024+7)
        binary_logits = self.binary_head(binary_input)             # (B, 2)

        return binary_logits, cat_logits


In [7]:
# Pre-cache encoder config + weights in CPU RAM for fast trial initialization
# (avoids from_pretrained disk I/O + config parsing per Optuna trial)
_encoder_config = AutoConfig.from_pretrained(MODEL_CHECKPOINT, cache_dir=CACHE_DIR)
_encoder_init_weights = AutoModel.from_pretrained(MODEL_CHECKPOINT, cache_dir=CACHE_DIR).cpu().state_dict()
print(f"Encoder weights cached in RAM ({sum(v.numel() * v.element_size() for v in _encoder_init_weights.values()) / 1e6:.1f} MB)")

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: albert/albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoder weights cached in RAM (70.7 MB)


In [8]:
# --- MULTI-TASK TRAINER ---
from types import SimpleNamespace
import torch.nn.functional as F

class MultiTaskTrainer(Trainer):
    """Joint training: focal CE on binary + BCE on categories.
    Uses WeightedRandomSampler to balance the ~10:1 class imbalance."""

    def __init__(self, focal_alpha, focal_gamma, aux_weight,
                 weighted_sampler=None, **kwargs):
        super().__init__(**kwargs)
        # Store focal params directly — inline CE avoids one_hot + sigmoid + BCE overhead
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        self.cat_loss_fn = nn.BCEWithLogitsLoss()
        self.aux_weight = aux_weight
        self.weighted_sampler = weighted_sampler

    def _focal_ce(self, logits, labels):
        """Focal cross-entropy — single fused kernel, no one_hot needed."""
        ce = F.cross_entropy(logits, labels, reduction="none")   # (B,)
        pt = torch.exp(-ce)                                       # P(correct class)
        return (self.focal_alpha * (1 - pt) ** self.focal_gamma * ce).mean()

    def get_train_dataloader(self):
        """Override to inject WeightedRandomSampler for class balancing."""
        if self.weighted_sampler is None:
            return super().get_train_dataloader()
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.per_device_train_batch_size,
            sampler=self.weighted_sampler,
            collate_fn=self.data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
            persistent_workers=self.args.dataloader_persistent_workers,
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        cat_labels = inputs.pop("cat_labels")

        binary_logits, cat_logits = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            token_type_ids=inputs.get("token_type_ids"),
        )

        loss_binary = self._focal_ce(binary_logits, labels)
        loss_cat = self.cat_loss_fn(cat_logits, cat_labels)
        loss = loss_binary + self.aux_weight * loss_cat

        if return_outputs:
            return loss, SimpleNamespace(logits=binary_logits)
        return loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        """Override to handle multi-task forward during evaluation."""
        with torch.no_grad():
            labels = inputs.pop("labels").to(self.args.device)
            cat_labels = inputs.pop("cat_labels").to(self.args.device)
            inputs = self._prepare_inputs(inputs)

            binary_logits, cat_logits = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                token_type_ids=inputs.get("token_type_ids"),
            )

            loss_binary = self._focal_ce(binary_logits, labels)
            loss_cat = self.cat_loss_fn(cat_logits, cat_labels)
            loss = loss_binary + self.aux_weight * loss_cat

        if prediction_loss_only:
            return (loss, None, None)
        return (loss, binary_logits, labels)


def compute_metrics_binary(pred):
    """Binary F1 for the positive (PCL) class."""
    preds = pred.predictions.argmax(-1)
    labels = pred.label_ids
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1, zero_division=0
    )
    return {"f1": f1, "precision": p, "recall": r, "accuracy": accuracy_score(labels, preds)}

In [9]:
# --- CUSTOM DATA COLLATOR (handles cat_labels alongside tokens) ---

class MultiTaskCollator(DataCollatorWithPadding):
    """Extends DataCollatorWithPadding to also collate cat_labels.
    Uses torch.stack for pre-tensorized labels (faster than torch.tensor from lists)."""
    def __call__(self, features):
        cat_labels = [f.pop("cat_labels") for f in features]
        batch = super().__call__(features)
        batch["cat_labels"] = torch.stack(cat_labels)  # stack pre-built tensors
        return batch

print(f"Batch: {BATCH_SIZE} x {GRAD_ACCUMULATION} accum = {BATCH_SIZE * GRAD_ACCUMULATION} effective")
print(f"Early stopping patience: 3 epochs. Cosine LR with 10% warmup.")

Batch: 32 x 2 accum = 64 effective
Early stopping patience: 3 epochs. Cosine LR with 10% warmup.


In [ ]:
import optuna, gc, json, shutil
from optuna.pruners import MedianPruner
import os
import time

def _cleanup_trial(trial_model, trial_trainer, trial_number):
    """Aggressively free all GPU memory from a trial."""
    if hasattr(trial_trainer, 'optimizer') and trial_trainer.optimizer is not None:
        trial_trainer.optimizer.zero_grad(set_to_none=True)
        del trial_trainer.optimizer
    if hasattr(trial_trainer, 'lr_scheduler'):
        del trial_trainer.lr_scheduler
    if hasattr(trial_trainer, 'accelerator'):
        trial_trainer.accelerator.free_memory()

    trial_model.cpu()
    del trial_model
    del trial_trainer

    gc.collect()
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    trial_dir = f"./results_optuna/trial_{trial_number}"
    if os.path.exists(trial_dir):
        shutil.rmtree(trial_dir, ignore_errors=True)

def _make_fresh_encoder():
    """Build encoder from cached config + weights (no disk I/O)."""
    encoder = AutoModel.from_config(_encoder_config)
    encoder.load_state_dict(_encoder_init_weights)
    return encoder

# --- Optuna Objective ---
def objective(trial):
    lr = trial.suggest_float("lr", 5e-6, 5e-5, log=True)
    focal_alpha = trial.suggest_float("focal_alpha", 0.25, 0.75)
    focal_gamma = trial.suggest_float("focal_gamma", 0.5, 3.0)
    aux_weight = trial.suggest_float("aux_weight", 0.05, 0.5)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    weight_decay = trial.suggest_float("weight_decay", 0.001, 0.1, log=True)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.05, 0.2)
    batch_size = trial.suggest_categorical("batch_size", [8, 16])
    grad_accum = 8 if batch_size == 8 else 4   # effective batch = 64

    # Build model from cached encoder (skips from_pretrained disk I/O)
    trial_model = MultiTaskModel(
        encoder=_make_fresh_encoder(),
        num_categories=NUM_CATEGORIES,
        dropout=dropout,
    ).to(DEVICE)

    trial_collator = MultiTaskCollator(tokenizer, padding="longest")

    class OptunaPruneCallback(EarlyStoppingCallback):
        def __init__(self, trial, patience):
            super().__init__(early_stopping_patience=patience)
            self.trial = trial

        def on_evaluate(self, args, state, control, metrics=None, **kwargs):
            super().on_evaluate(args, state, control, metrics=metrics, **kwargs)
            f1 = metrics.get("eval_f1", 0.0)
            self.trial.report(f1, step=int(state.epoch))
            if self.trial.should_prune():
                raise optuna.TrialPruned()

    trial_trainer = MultiTaskTrainer(
        focal_alpha=focal_alpha,
        focal_gamma=focal_gamma,
        aux_weight=aux_weight,
        weighted_sampler=sampler,
        model=trial_model,
        args=TrainingArguments(
            output_dir=f"./results_optuna/trial_{trial.number}",
            eval_strategy="epoch",
            save_strategy="no",
            load_best_model_at_end=False,
            learning_rate=lr,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size * 2,
            gradient_accumulation_steps=grad_accum,
            num_train_epochs=12,
            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,
            lr_scheduler_type="cosine",
            metric_for_best_model="f1",
            greater_is_better=True,
            label_names=["labels", "cat_labels"],
            remove_unused_columns=False,
            logging_strategy="epoch",
            fp16=True,
            torch_empty_cache_steps=50,          # periodic VRAM defrag
            dataloader_num_workers=4,
            dataloader_pin_memory=True,
            dataloader_persistent_workers=False,  # workers pin GPU-bound memory
            optim="adamw_torch_fused",
        ),
        data_collator=trial_collator,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics_binary,
        callbacks=[OptunaPruneCallback(trial, patience=3)],
    )

    try:
        trial_trainer.train()
        eval_f1s = [log["eval_f1"] for log in trial_trainer.state.log_history if "eval_f1" in log]
        best_f1 = max(eval_f1s) if eval_f1s else 0.0
    except Exception as e:
        best_f1 = 0.0
        raise
    finally:
        _cleanup_trial(trial_model, trial_trainer, trial.number)

    return best_f1

In [11]:
# --- Free VRAM & Run Study ---
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Delete original model/trainer/predictions still in GPU memory from earlier cells
for _var in ['model', 'trainer', 'mt_collator', 'preds_output', 'raw_logits',
             'probs', 'default_preds', 'final_preds', 'true_labels', 'preds_t']:
    if _var in globals():
        del globals()[_var]
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, "
      f"{torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

study = optuna.create_study(
    direction="maximize",
    study_name="pcl_multitask_f1",
    pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=2),
)

[I 2026-03-02 01:07:35,457] A new study created in memory with name: pcl_multitask_f1


VRAM after cleanup: 0.00 GB allocated, 0.00 GB reserved


In [12]:
print("Starting Optuna hyperparameter search (20 trials, max 12 epochs each, patience=3)...")
study.optimize(objective, n_trials=20, show_progress_bar=True)

Starting Optuna hyperparameter search (20 trials, max 12 epochs each, patience=3)...


  0%|          | 0/20 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


[W 2026-03-02 01:07:42,942] Trial 0 failed with parameters: {'lr': 5.2321935152354785e-06, 'focal_alpha': 0.7205684773128462, 'focal_gamma': 0.6084596869055698, 'aux_weight': 0.3507739036111253, 'dropout': 0.19158630143813105, 'weight_decay': 0.05886785595789415, 'warmup_ratio': 0.0741889866165048, 'batch_size': 32} because of the following error: OutOfMemoryError('CUDA out of memory. Tried to allocate 60.00 MiB. GPU 0 has a total capacity of 7.65 GiB of which 51.31 MiB is free. Process 4409 has 29.41 MiB memory in use. Process 335641 has 12.66 MiB memory in use. Process 335914 has 37.41 MiB memory in use. Process 335992 has 37.41 MiB memory in use. Process 565905 has 21.41 MiB memory in use. Including non-PyTorch memory, this process has 6.91 GiB memory in use. Of the allocated memory 6.73 GiB is allocated by PyTorch, and 21.87 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragme

OutOfMemoryError: CUDA out of memory. Tried to allocate 60.00 MiB. GPU 0 has a total capacity of 7.65 GiB of which 51.31 MiB is free. Process 4409 has 29.41 MiB memory in use. Process 335641 has 12.66 MiB memory in use. Process 335914 has 37.41 MiB memory in use. Process 335992 has 37.41 MiB memory in use. Process 565905 has 21.41 MiB memory in use. Including non-PyTorch memory, this process has 6.91 GiB memory in use. Of the allocated memory 6.73 GiB is allocated by PyTorch, and 21.87 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# --- Results ---
print(f"\n{'='*60}")
print(f"Best trial: #{study.best_trial.number}")
print(f"  Best F1: {study.best_value:.4f}")
print(f"  Params:")
for k, v in study.best_params.items():
    print(f"    {k}: {v}")

# Save study results
results_df = study.trials_dataframe()
results_df.to_csv("optuna_span_results.csv", index=False)
with open("best_hyperparams.json", "w") as f:
    json.dump({"best_f1": study.best_value, **study.best_params}, f, indent=2)
print(f"\nStudy results saved to optuna_span_results.csv")
print(f"Best hyperparams saved to best_hyperparams.json")

In [ ]:
# --- RETRAIN WITH BEST HYPERPARAMS & SAVE ---
bp = study.best_params
print(f"Retraining with best hyperparams (F1={study.best_value:.4f})...")
print(json.dumps(bp, indent=2))

In [ ]:
# Build final model with best dropout
final_model = MultiTaskModel(
    MODEL_CHECKPOINT,
    num_categories=NUM_CATEGORIES,
    dropout=bp["dropout"],
    cache_dir=CACHE_DIR,
).to(DEVICE)
final_model = torch.compile(final_model, dynamic=True)

final_batch = int(bp["batch_size"])
final_accum = 4 if final_batch == 16 else 2  # effective batch ~64

final_collator = MultiTaskCollator(tokenizer, padding="longest")

final_trainer = MultiTaskTrainer(
    focal_alpha=bp["focal_alpha"],
    focal_gamma=bp["focal_gamma"],
    aux_weight=bp["aux_weight"],
    weighted_sampler=sampler,
    model=final_model,
    args=TrainingArguments(
        output_dir="./results_multitask",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=3,
        learning_rate=bp["lr"],
        per_device_train_batch_size=final_batch,
        per_device_eval_batch_size=final_batch * 2,
        gradient_accumulation_steps=final_accum,
        num_train_epochs=12,
        weight_decay=bp["weight_decay"],
        warmup_ratio=bp["warmup_ratio"],
        lr_scheduler_type="cosine",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        label_names=["labels", "cat_labels"],
        remove_unused_columns=False,
        logging_strategy="steps",
        logging_steps=50,
        fp16=True,
        torch_compile=False,                     # already compiled manually above
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        dataloader_persistent_workers=True,
        optim="adamw_torch_fused",
    ),
    data_collator=final_collator,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics_binary,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [ ]:
print(f"\nTraining final model (12 epochs max, patience=3)...")
final_trainer.train()

In [ ]:
# --- Threshold Optimization ---
print("\n--- Threshold Optimization ---")
preds_output = final_trainer.predict(val_ds)
raw_logits = torch.tensor(preds_output.predictions)
probs = torch.softmax(raw_logits, dim=-1)[:, 1].numpy()
true_labels = preds_output.label_ids

best_f1, best_thresh = 0, 0.5
for t in np.arange(0.20, 0.80, 0.01):
    preds_t = (probs >= t).astype(int)
    _, _, f1_t, _ = precision_recall_fscore_support(true_labels, preds_t, average="binary", pos_label=1, zero_division=0)
    if f1_t > best_f1:
        best_f1, best_thresh = f1_t, t

final_preds = (probs >= best_thresh).astype(int)
p, r, f1, _ = precision_recall_fscore_support(true_labels, final_preds, average="binary", pos_label=1, zero_division=0)
acc = accuracy_score(true_labels, final_preds)

default_preds = (probs >= 0.5).astype(int)
_, _, f1_default, _ = precision_recall_fscore_support(true_labels, default_preds, average="binary", pos_label=1, zero_division=0)

print(f"\n  Default threshold (0.50):  F1={f1_default:.4f}")
print(f"  Optimal threshold ({best_thresh:.2f}):  F1={f1:.4f}  P={p:.4f}  R={r:.4f}  Acc={acc:.4f}")
gain = (f1 - f1_default) * 100
print(f"  → Threshold tuning: {'+' if gain > 0 else ''}{gain:.2f} F1 pts")
print("  TARGET BEAT!" if f1 > 0.51 else "  Needs further tuning.")

# --- Save best model ---
orig_model = final_model._orig_mod if hasattr(final_model, '_orig_mod') else final_model
orig_model.encoder.save_pretrained("./pcl_multitask_model")
tokenizer.save_pretrained("./pcl_multitask_model")
torch.save({
    "model_state_dict": orig_model.state_dict(),
    "optimal_threshold": best_thresh,
    "best_f1": best_f1,
    "best_hyperparams": bp,
}, "./pcl_multitask_model/full_model.pt")

# Also save hyperparams alongside model config
with open("./pcl_multitask_model/best_hyperparams.json", "w") as f:
    json.dump({"best_f1": float(best_f1), "optimal_threshold": float(best_thresh), **bp}, f, indent=2)

print(f"\nFinal model + threshold + hyperparams saved to ./pcl_multitask_model/")